<div dir="rtl" style="text-align:right"><h1>שיעור 3, חלק 1: מנוירון לייצוג – איך מחשב לומד</h1>
<p>יישומי בינה מלאכותית AI | מחברת להתנסות. <b>אין חובת הגשת מחברת.</b></p>
<ol>
<li><b>נוירון אחד</b> – סכום משוקלל וסף, ביד.</li>
<li><b>רשת קטנה לומדת</b> – אימון על נתוני מאפייה בדויים: השגיאה יורדת. נשנה את קצב הלמידה ואת גודל הרשת, ונספור את הפרמטרים.</li>
<li><b>ייצוגים</b> – מספרים אמיתיים ממודל ייצוגים פתוח, למילים שאתם בוחרים.</li>
</ol>
<p><b>בכל סעיף:</b> ① נחשו מה יקרה ② הריצו ③ שנו דבר אחד ④ הריצו שוב וראו מה השתנה.</p>
<p>מריצים תא אחרי תא בלחיצה על ▶. מעבד רגיל (<code>CPU</code>) מספיק; אין צורך ב-<code>GPU</code>, במפתח <code>API</code> או במנוי. סעיף 3 מוריד מודל פתוח של כ-470MB. אזהרה על <code>HF_TOKEN</code> – אפשר להתעלם.</p></div>

In [ ]:
# Setup. Everything below is preinstalled on Google Colab.
import warnings

import matplotlib.pyplot as plt
import numpy as np
import sklearn
import torch
import transformers
from sklearn.exceptions import ConvergenceWarning
from sklearn.neural_network import MLPClassifier

warnings.filterwarnings("ignore", category=ConvergenceWarning)
print("scikit-learn", sklearn.__version__, "| transformers", transformers.__version__, "| torch", torch.__version__)


def rtl(text):
    # matplotlib draws left to right; reversing a Hebrew word displays it correctly.
    return text[::-1] if any("\u0590" <= ch <= "\u05ff" for ch in text) else text

<div dir="rtl" style="text-align:right"><h2>1 | נוירון אחד</h2>
<p><b>נוירון</b> (בתוכנה) הוא חישוב פשוט: כל קלט מוכפל ב<b>משקל</b>, מחברים הכול, מוסיפים <b>הטיה</b> (<code>bias</code>), ובודקים אם הסכום עבר סף (כאן: 0). השאלה: האם המאפייה צריכה לאפות עוד מגש עוגיות למחר?</p>
<ul>
<li>הזמנות מראש למחר – משקל חיובי: כל הזמנה דוחפת לאפות.</li>
<li>מגשים שנשארו מהיום – משקל שלילי: כל מגש שנשאר מושך לא לאפות.</li>
</ul>
<blockquote>① נחשו: עם 6 הזמנות ו-3 מגשים שנשארו, משקלים 0.5 ו-<span dir="ltr">−1</span> והטיה <span dir="ltr">−2</span> – לאפות או לא?</blockquote></div>

In [ ]:
def neuron(inputs, weights, bias):
    total = sum(x * w for x, w in zip(inputs, weights)) + bias
    return total, (1 if total >= 0 else 0)


weights = [0.5, -1.0]  # orders, leftover trays (chosen by hand)  <- ③ change one weight
bias = -2.0
for orders, leftover in [(6, 3), (10, 1), (4, 0)]:
    total, decision = neuron([orders, leftover], weights, bias)
    print(f"orders={orders:2}  leftover={leftover}  ->  sum={total:5.1f}  ->  {'bake another tray' if decision else 'no extra tray'}")

<div dir="rtl" style="text-align:right"><blockquote>③ שנו משקל אחד (למשל את המשקל של ההזמנות ל-<code>1.0</code>) והריצו שוב. ④ איזה מקרה התהפך, ולמה? כאן המשקלים נבחרו ביד; בסעיף 2 <b>האימון</b> יקבע אותם.</blockquote></div>

<div dir="rtl" style="text-align:right"><h2>2 | רשת קטנה לומדת מדוגמאות</h2>
<p><b>רשת נוירונים</b> היא שכבות של נוירונים כאלה: הפלט של שכבה אחת הוא הקלט של הבאה. <b>אימון</b> (כמו בשיעור 2) פירושו לשנות את המשקלים צעד אחרי צעד, כך שהשגיאה על דוגמאות האימון קטנה. כל המספרים שהאימון קובע – המשקלים וההטיות – נקראים <b>פרמטרים</b>.</p>
<p>הנתונים <b>בדויים</b>: כלל פשוט עם מעט רעש קובע אם היה צריך עוד מגש. נשתמש ב-<code>MLPClassifier</code> של <code>scikit-learn</code>: רשת עם שכבה נסתרת אחת.</p>
<blockquote>① נחשו: אחרי צעד אימון אחד, הרשת כבר תצדק ברוב הדוגמאות? ואחרי 300?</blockquote></div>

In [ ]:
rng = np.random.default_rng(3)
orders = rng.integers(0, 21, 160).astype(float)
leftover = rng.integers(0, 11, 160).astype(float)
needed = (orders - 1.5 * leftover + rng.normal(0, 1.5, 160) > 5).astype(int)  # made-up rule + noise
X = np.column_stack([orders / 20, leftover / 10])  # scaled to 0..1
X_train, y_train, X_test, y_test = X[:120], needed[:120], X[120:], needed[120:]


def train(hidden=8, learning_rate=0.005, steps=300, report=True):
    """Train step by step (one pass over the 120 examples per step)."""
    net = MLPClassifier(hidden_layer_sizes=(hidden,), learning_rate_init=learning_rate, max_iter=1, warm_start=True, random_state=0)
    losses, snapshots, train_acc = [], {}, {}
    for step in range(1, steps + 1):
        net.fit(X_train, y_train)
        losses.append(net.loss_)
        if step in (1, steps):
            snapshots[step] = (net.coefs_[0].copy(), net.intercepts_[0].copy(), net.coefs_[1].copy(), net.intercepts_[1].copy())
            train_acc[step] = net.score(X_train, y_train)
        if report and step in (1, 50, 100, 200, 300):
            print(f"step {step:3}: loss {net.loss_:.3f}   accuracy on training examples {net.score(X_train, y_train):.0%}")
    return net, losses, snapshots, train_acc


net, losses, snapshots, train_acc = train()  # the baseline: 8 hidden neurons, learning rate 0.005
print(f"loss after 1 step: {losses[0]:.3f}   after 300 steps: {losses[-1]:.3f}")
print(f"accuracy on the training examples: {train_acc[1]:.0%} after 1 step, {train_acc[300]:.0%} after 300")
print(f"accuracy on 40 examples the network did not train on: {net.score(X_test, y_test):.0%}")

In [ ]:
def predict_with(snapshot, points):
    w1, b1, w2, b2 = snapshot
    hidden = np.maximum(0, points @ w1 + b1)  # ReLU
    return (1 / (1 + np.exp(-(hidden @ w2 + b2)))).ravel()


grid_x, grid_y = np.meshgrid(np.linspace(0, 1, 120), np.linspace(0, 1, 120))
grid = np.column_stack([grid_x.ravel(), grid_y.ravel()])

fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))
axes[0].plot(losses)
axes[0].set_title("loss (error) during training")
axes[0].set_xlabel("training steps")
for ax, step in zip(axes[1:], (1, 300)):
    p = predict_with(snapshots[step], grid).reshape(grid_x.shape)
    ax.contourf(grid_x * 20, grid_y * 10, p, levels=[0, 0.5, 1], colors=["#f5d9c8", "#cee3da"], alpha=0.8)
    ax.scatter(X_train[:, 0] * 20, X_train[:, 1] * 10, c=np.where(y_train, "#2d5d4f", "#c97852"), s=18)
    ax.set_title(f"decision after {step} step(s)")
    ax.set_xlabel("orders for tomorrow")
    ax.set_ylabel("trays left today")
plt.tight_layout()
plt.show()

<div dir="rtl" style="text-align:right"><p>משמאל: השגיאה יורדת לאורך האימון. באמצע ומימין: האזור הירוק הוא «לאפות עוד מגש». אחרי צעד אחד הרשת עוד כמעט לא מבחינה בין המקרים; אחרי 300 צעדים הקו מפריד את רוב הנקודות (לא את כולן – יש רעש בנתונים). אף אחד לא כתב את הכלל ביד – האימון כיוונן את המשקלים.</p>
<h3>2א | משנים דבר אחד: קצב הלמידה וגודל הרשת</h3>
<p><b>קצב הלמידה</b> קובע כמה גדול כל צעד אימון. <b>גודל השכבה הנסתרת</b> קובע כמה נוירונים יש בה.</p>
<blockquote>① נחשו: קצב למידה קטן פי 10 – השגיאה אחרי 300 צעדים תהיה גבוהה יותר או נמוכה יותר? ומה עם 2 נוירונים במקום 8?</blockquote>
<p>③ בתא הבא כל שורה משנה <b>דבר אחד</b> לעומת שורת הבסיס. הוסיפו שורה משלכם והריצו שוב.</p></div>

In [ ]:
print(f"{'hidden':>6} {'learning rate':>13} {'loss after 300':>14} {'train acc':>9} {'test acc':>8}")
for hidden, lr in [(8, 0.005), (8, 0.0005), (8, 0.05), (2, 0.005), (32, 0.005)]:  # <- ③ add your own row
    n, l, _, acc = train(hidden=hidden, learning_rate=lr, report=False)
    print(f"{hidden:6} {lr:13} {l[-1]:14.3f} {acc[300]:9.0%} {n.score(X_test, y_test):8.0%}")

<div dir="rtl" style="text-align:right"><p>④ שימו לב: קצב קטן מדי – הרשת כמעט לא מתקדמת ב-300 צעדים. גדול יותר – מתקדמת מהר יותר (ולא תמיד טוב יותר). שנו בכל שורה רק דבר אחד, כמו בשיעור 2.</p>
<h3>2ב | כמה פרמטרים יש ברשת?</h3>
<p>כל חיבור בין נוירונים הוא משקל, ולכל נוירון יש הטיה. עם 2 קלטים, 8 נוירונים נסתרים ופלט אחד: 2×8 משקלים + 8 הטיות + 8×1 משקלים + הטיה אחת.</p>
<blockquote>① נחשו את המספר לפני שמריצים.</blockquote></div>

In [ ]:
def count_parameters(net):
    return sum(w.size for w in net.coefs_) + sum(b.size for b in net.intercepts_)


print("baseline network (8 hidden):", count_parameters(net), "parameters")  # 2*8 + 8 + 8*1 + 1
for hidden in (2, 32):  # <- ③ try another size
    n, _, _, _ = train(hidden=hidden, report=False)
    print(f"{hidden} hidden:", count_parameters(n), "parameters")
print("For comparison, the small language model in part 2 has about 0.49 billion (its notebook counts them).")

<div dir="rtl" style="text-align:right"><h2>3 | ייצוגים (embeddings): מילים כמספרים</h2>
<p>רשת עובדת רק עם מספרים. לכן כל מילה מקבלת <b>ייצוג</b>: רשימה ארוכה של מספרים, שנלמדת באימון. למילים שמופיעות בהקשרים דומים יש ייצוגים קרובים. נשתמש במודל הייצוגים הפתוח <code>sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2</code> – אותו מודל שבמפה באתר. הוא אומן גם על זוגות של משפטים ותרגומם, ולכן מילה ותרגומה יוצאות קרובות מאוד. לכל טקסט הוא נותן 384 מספרים.</p>
<p><b>דמיון קוסינוס</b> משווה שתי רשימות: ככל שהמספר גבוה יותר – הן דומות יותר (1 – כיוון זהה). גם מילים לא קשורות מקבלות כאן מספר (בערך 0.3–0.5), ולכן משווים מספרים זה לזה ולא לאפס.</p>
<blockquote>① נחשו: מי הכי קרוב ל«לחם» – «bread», «עוגה», «קפה» או «שמח»?</blockquote></div>

In [ ]:
from transformers import AutoModel, AutoTokenizer

EMB_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
emb_tok = AutoTokenizer.from_pretrained(EMB_MODEL)
emb_model = AutoModel.from_pretrained(EMB_MODEL)
emb_model.eval()


def embed(texts):
    batch = emb_tok(texts, padding=True, return_tensors="pt")
    with torch.no_grad():
        out = emb_model(**batch).last_hidden_state
    mask = batch["attention_mask"].unsqueeze(-1).float()
    return ((out * mask).sum(1) / mask.sum(1)).numpy()  # mean pooling: one vector per text


def cosine(a, b):
    va, vb = embed([a, b])
    return float(va @ vb / (np.linalg.norm(va) * np.linalg.norm(vb)))


print("numbers per word:", embed(["לחם"]).shape[1])
for a, b in [("לחם", "bread"), ("לחם", "עוגה"), ("לחם", "קפה"), ("לחם", "שמח"), ("מחר", "tomorrow"), ("שמח", "עצוב")]:
    print(f"{a:>6} ~ {b:<8}  cosine = {cosine(a, b):.2f}")

<div dir="rtl" style="text-align:right"><p>④ «שמח» ו«עצוב» הפוכים במשמעות, ובכל זאת קרובים למדי – שתיהן מילים של רגש שמופיעות בהקשרים דומים. ייצוגים משקפים <b>הקשרים שבהם מילים מופיעות</b>, לא הגדרה של משמעות.</p>
<h3>3א | המילים שלכם</h3>
<p>③ כתבו ברשימה מילים משלכם, בעברית או באנגלית (8–20 מילים). התא ימצא לכל מילה את שתי השכנות הקרובות לה <b>מתוך הרשימה שלכם</b>, ויצייר מפה בשני ממדים.</p>
<blockquote>① לפני ההרצה: אילו מילים לדעתכם יתקבצו יחד?</blockquote></div>

In [ ]:
my_words = ["לחם", "עוגה", "עוגיות", "bread", "cake", "קפה", "תה", "coffee", "tea",
            "מחר", "אתמול", "tomorrow", "שמח", "עצוב", "happy", "אמא", "אבא", "mother"]  # <- ③ your own words

vectors = embed(my_words)
unit = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
sim = unit @ unit.T
for i, w in enumerate(my_words):
    order = [j for j in np.argsort(-sim[i]) if j != i][:2]
    print(f"{w:>9}: " + ", ".join(f"{my_words[j]} ({sim[i, j]:.2f})" for j in order))

centered = vectors - vectors.mean(0)
_, singular, vt = np.linalg.svd(centered, full_matrices=False)
xy = centered @ vt[:2].T
print(f"\nthe 2D map keeps {(singular[:2] ** 2).sum() / (singular ** 2).sum():.0%} of the variance")
plt.figure(figsize=(7, 5.5))
plt.scatter(xy[:, 0], xy[:, 1], color="#4a6c7a")
for (x, y), w in zip(xy, my_words):
    plt.annotate(rtl(w), (x, y), textcoords="offset points", xytext=(4, 4))
plt.title("PCA to 2D - distances are approximate")
plt.show()

<div dir="rtl" style="text-align:right"><p>④ <b>המפה דוחסת</b> 384 מספרים לשניים, ולכן המרחקים בה משוערים. השכנות שהודפסו מעל המפה מחושבות במלוא 384 המספרים – בהן סומכים.</p>
<h2>4 | מה ראינו</h2>
<ul>
<li><b>נוירון:</b> סכום משוקלל, הטיה וסף. המשקלים קובעים מה חשוב.</li>
<li><b>אימון:</b> משנים משקלים צעד אחרי צעד כדי להקטין שגיאה; קצב הלמידה קובע את גודל הצעד. אחרי האימון הפרמטרים קבועים, ומשתמשים ברשת לחיזוי.</li>
<li><b>ייצוגים:</b> מילה הופכת לרשימת מספרים; דמיון בין רשימות משקף הקשרים שנלמדו – כולל מוזרויות.</li>
</ul>
<p>בחלק 2 נראה איך מודל שפה בונה על כל אלה כדי לכתוב טקסט. חזרו לאתר להמשך.</p></div>